In [14]:
import torch
import torch.nn as nn


In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [16]:
cross = nn.CrossEntropyLoss

In [17]:
def myonedCrossEntropyLoss(logits, target):
    prob = torch.exp(logits)
    prob = prob/prob.sum()
    return -torch.log(prob(target))

In [30]:
def myCrossEntropyLoss(logits, target):
    C = logits.shape[1]
    flat_logits = logits.movedim(1, -1).reshape(-1, C)
    flat_target = target.reshape(-1)
    losses = [myonedCrossEntropyLoss(flat_logits[i], flat_target[i])
              for i in range(len(flat_target))]
    return torch.stack(losses).mean()

In [31]:
mycross = myCrossEntropyLoss

In [32]:
torch.manual_seed(0)
C = 5
shapes = [((8, C), (8,)),                    # dim 1
          ((8, C, 7), (8, 7)),               # dim 2
          ((8, C, 6, 7), (8, 6, 7)),         # dim 3
          ((4, C, 3, 6, 7), (4, 3, 6, 7))]   # dim 4

for logits_shape, target_shape in shapes:
    logits = torch.randn(*logits_shape, requires_grad=True)
    target = torch.randint(0, C, target_shape)

    mine = myCrossEntropyLoss(logits, target)
    ref  = nn.CrossEntropyLoss()(logits, target)

    assert torch.allclose(mine, ref, atol=1e-6), f"mismatch at dim {target.dim()}"
    mine.backward()   # confirm gradients flow
    print(f"dim {target.dim()}: diff={mine.item()- ref.item():.6f} ")

dim 1: diff=0.000000 
dim 2: diff=0.000000 
dim 3: diff=-0.000000 
dim 4: diff=0.000000 
